In [1]:
from common_setup import *
from pathos.multiprocessing import ProcessingPool as Pool


In [2]:
from InitializeSpeciesPool import *
from LV import *
from VariousMetrics import *

########################################################
########################################################
# Fig 1-1
########################################################
########################################################

##~~~~~~~~~~~~~~ This figure is cartoon ~~~~~~~~~~~~~~##

In [3]:
def estimate_mean_ratio(capacityVar, num_samples=500000, seed=None):
    """
    Estimate E[U/V] where U and V are independent and U = max(N(1, capacityVar), 0.1).

    Parameters:
        capacityVar (float): Variance of the normal distribution.
        num_samples (int): Number of samples to generate for the estimation.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        float: Estimated expected value of U/V.
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Define the mean and standard deviation
    mu = 1
    sigma = np.sqrt(capacityVar)
    
    # Generate samples from N(1, capacityVar)
    U_samples = np.random.normal(mu, sigma, num_samples)
    V_samples = np.random.normal(mu, sigma, num_samples)
    
    # Apply the transformation f_k = max(X, 0.1)
    U_samples = np.maximum(U_samples, 0.1)
    V_samples = np.maximum(V_samples, 0.1)
    
    # Compute the ratios
    ratios = U_samples / V_samples
    
    # Calculate the mean of the ratios
    mean_ratio = np.mean(ratios)
    
    return mean_ratio

estimate_mean_ratio(0)

1.0

In [6]:
import os
import json
from tqdm import tqdm

# Store the original u_list to reuse in each iteration
original_u_list = np.arange(0, 1.2, 0.1)

for var_k in [0.05, 0.1, 0.15, 0.2, 0.25]:
    session_name = f"Simulation_Data/new_k_gamma_{var_k}_defined_pool_nooverlap_12from48"
    if not os.path.exists(session_name):
        os.makedirs(session_name)
        print("The new directory is created!")

    # Simulation parameters
    N_simul = 8
    S = 12
    # Create a fresh copy of u_list for each iteration
    u_list = original_u_list.copy()
    o = 0
    t = [0, 5000]
    num_C = 9
    num_S = 12
    N = 48
    threshold = 1e-3

    if var_k==0:
        f_k= lambda: 1
    else:
        f_k = lambda: max(np.random.gamma(shape=1/var_k, scale=var_k, size=1)[0], 0.1)

    # Normalize u_list using estimate_mean_ratio
    estimated_mean_ratio = estimate_mean_ratio(var_k)
    print("estimated_mean_ratio", estimated_mean_ratio)

    u_list = u_list / estimated_mean_ratio

    Intention_to_reset = True
    if Intention_to_reset or not os.path.isfile(session_name + '/Community.json'):
        tasks = []
        for i, u in enumerate(u_list):
            for itt in range(N_simul):
                tasks.append((i, itt, u, session_name, t, N, o, threshold, num_C, num_S, f_k))

        def simulate_task(task):
            i, itt, u, session_name, t, N, o, threshold, num_C, num_S, f_k = task
            np.random.seed(itt)
            f_interaction = lambda: uniform_distribution(u, o)
            I, g, k = InitializeSpeceiesPool(N, f_interaction, f_g=lambda: np.ones(1),
                                               f_k=f_k, is_diagonal_one=True, save_path=session_name)
            CommunitiesLibrary = InitializeRandomCommunityPool(N, num_C, num_S, I, g, k, save_path=session_name)
            y = np.random.rand(N) * 0.1
            sc_list = {}
            for idx in range(num_C):
                y1 = run_lotka_volterra(y, t, CommunitiesLibrary[idx, :], I, g, k)
                y1[y1 < threshold] = 0
                sc_list[idx] = y1.tolist()
            cc_list = {}
            for idx in range(num_C):
                for jdx in range(idx+1, num_C):
                    y1 = np.array(sc_list[idx])
                    y2 = np.array(sc_list[jdx])
                    y3 = (y1 + y2) / 2
                    survived = y3 > threshold
                    y3 = run_lotka_volterra(y3, t, survived, I, g, k)
                    y3[y3 < threshold] = 0
                    cc_list[f"{idx}_{jdx}"] = y3.tolist()
                    
            return (u, itt, sc_list, cc_list)

        with Pool(processes=8) as pool:
            results = list(tqdm(pool.imap(simulate_task, tasks), total=len(tasks)))

        # Aggregate results into a dictionary
        all_results = {}
        for u, itt, sc_list, cc_list, in results:
            if u not in all_results:
                all_results[u] = {}
            all_results[u][f"community_{itt}"] = {"sc_list": sc_list, "cc_list": cc_list}

        with open(session_name + '/Community.json', 'w') as f:
            json.dump(all_results, f, indent=4)
        print(f"Data successfully saved to {session_name + '/Community.json'}!")


The new directory is created!
estimated_mean_ratio 1.0609175642487034
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.

  0%|          | 0/96 [00:00<?, ?it/s]



Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.




Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.

  1%|          | 1/96 [00:17<27:18, 17.24s/it]


Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.
Folder 'Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48' already exists.

...

Folder 'Simulation_Data/new_k_gamma_0.25_defined_pool_nooverlap_12from48' already exists.


 92%|█████████▏| 88/96 [02:30<00:08,  1.10s/it]

Folder 'Simulation_Data/new_k_gamma_0.25_defined_pool_nooverlap_12from48' already exists.


100%|██████████| 96/96 [02:40<00:00,  1.67s/it]

Data successfully saved to Simulation_Data/new_k_gamma_0.25_defined_pool_nooverlap_12from48/Community.json!
